##**Problem Statement**

Your objective is to explore the relationship between trader performance and market sentiment, uncover hidden patterns, and deliver insights that can drive smarter trading strategies.

### ***Importing Libraries***

In [77]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

###**Loading the data**

In [78]:
url1 = "https://drive.google.com/uc?id=1W9JOvI9XgcbF4_Rc4ctHECnI1PckJnhP"
url2 = "https://drive.google.com/uc?id=1gcNLnlbLGcVHzmEwkoCjgN0m7oGvNZFQ"

In [79]:
historical_data = pd.read_csv(url1)
market_sentiment = pd.read_csv(url2)

####**Understanding Historical Data**

In [80]:
historical_data.head()

,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,02-12-2024 22:50,1002.518996,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,1.730000e+12
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,BUY,02-12-2024 22:50,1146.558564,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050043,1.080000e+15,1.730000e+12
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,BUY,02-12-2024 22:50,1289.488521,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.003055,1.050000e+15,1.730000e+12


In [81]:
historical_data.shape

(211224, 16)

In [82]:
historical_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211224 entries, 0 to 211223
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Account           211224 non-null  object 
 1   Coin              211224 non-null  object 
 2   Execution Price   211224 non-null  float64
 3   Size Tokens       211224 non-null  float64
 4   Size USD          211224 non-null  float64
 5   Side              211224 non-null  object 
 6   Timestamp IST     211224 non-null  object 
 7   Start Position    211224 non-null  float64
 8   Direction         211224 non-null  object 
 9   Closed PnL        211224 non-null  float64
 10  Transaction Hash  211224 non-null  object 
 11  Order ID          211224 non-null  int64  
 12  Crossed           211224 non-null  bool   
 13  Fee               211224 non-null  float64
 14  Trade ID          211224 non-null  float64
 15  Timestamp         211224 non-null  float64
dtypes: bool(1), float64(

In [83]:
historical_data.isnull().sum()

,0
Account,0
Coin,0
Execution Price,0
Size Tokens,0
Size USD,0
Side,0
Timestamp IST,0
Start Position,0
Direction,0
Closed PnL,0


In [84]:
historical_data.duplicated().sum()

np.int64(0)

In [85]:
historical_data['Order ID'].nunique()

50555

In [86]:
len(historical_data)

211224

In [87]:
historical_data['Coin'].value_counts()

,count
Coin,
HYPE,68005
@107,29992
BTC,26064
ETH,11158
SOL,10691
...,...
@18,1
@30,1
@25,1


In [88]:
historical_data['Side'].value_counts()

,count
Side,
SELL,108528
BUY,102696


In [89]:
historical_data['Direction'].value_counts()

,count
Direction,
Open Long,49895
Close Long,48678
Open Short,39741
Close Short,36013
Sell,19902
Buy,16716
Spot Dust Conversion,142
Short > Long,70
Long > Short,57


####**Understanding Market Sentiment data**

In [90]:
market_sentiment.head()

,timestamp,value,classification,date
0,1517463000,30,Fear,2018-02-01
1,1517549400,15,Extreme Fear,2018-02-02
2,1517635800,40,Fear,2018-02-03
3,1517722200,24,Extreme Fear,2018-02-04
4,1517808600,11,Extreme Fear,2018-02-05


In [91]:
market_sentiment.shape

(2644, 4)

In [92]:
market_sentiment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2644 entries, 0 to 2643
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   timestamp       2644 non-null   int64 
 1   value           2644 non-null   int64 
 2   classification  2644 non-null   object
 3   date            2644 non-null   object
dtypes: int64(2), object(2)
memory usage: 82.8+ KB


In [93]:
market_sentiment.isnull().sum()

,0
timestamp,0
value,0
classification,0
date,0


In [94]:
market_sentiment.duplicated().sum()

np.int64(0)

###***Preprocesssing***

In [95]:
valid_directions = [
    'Open Long', 'Close Long',
    'Open Short', 'Close Short',
    'Buy', 'Sell'
]

In [96]:
historical_data = historical_data[historical_data['Direction'].isin(valid_directions)]

**I excluded system-generated and non-standard events such as Liquidated Isolated Short, Spot Dust Conversion, and Auto-Deleveraging because they do not represent intentional trading behavior and could distort performance analysis**

In [97]:
historical_data['Direction'].value_counts()

,count
Direction,
Open Long,49895
Close Long,48678
Open Short,39741
Close Short,36013
Sell,19902
Buy,16716


In [98]:
historical_data = historical_data[historical_data['Coin'] == 'BTC']

**Since the sentiment dataset represents Bitcoin market sentiment(as per the DS task document), I filtered the trading dataset to include only Bitcoin trades to ensure consistency and avoid cross-asset bias in the analysis.**

In [99]:
len(historical_data)

26044

In [100]:
historical_data['Order ID'].nunique()

5170

In [101]:
historical_data.columns

Index(['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side',
       'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL',
       'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID',
       'Timestamp'],
      dtype='object')

In [102]:
agg_df_bitcoin = historical_data.groupby(['Account', 'Order ID']).agg({
    'Execution Price': 'mean',
    'Size USD': 'sum',
    'Size Tokens': 'sum',
    'Fee': 'sum',
    'Closed PnL': 'sum',
    'Side': 'first',
    'Direction': 'first',
    'Timestamp IST': 'first'
}).reset_index()

**The dataset initially contained execution-level data, where each order was split into multiple executions. I identified this using Order ID uniqueness and aggregated the data at the order level.**

In [103]:
len(agg_df_bitcoin)

5170

In [104]:
agg_df_bitcoin.head()

,Account,Order ID,Execution Price,Size USD,Size Tokens,Fee,Closed PnL,Side,Direction,Timestamp IST
0,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,88138871892,93700.000000,3456.59,0.03689,0.345657,0.000,SELL,Open Short,23-04-2025 05:12
1,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,88141374064,93500.000000,3449.21,0.03689,0.344921,7.378,BUY,Close Short,23-04-2025 05:22
2,0x271b280974205ca63b716753467d5a371de622ab,85391103553,82533.666667,460268.75,5.57673,154.650290,0.000,SELL,Open Short,09-04-2025 23:51
3,0x271b280974205ca63b716753467d5a371de622ab,85391332312,82575.736842,780030.44,9.44631,262.090211,0.000,SELL,Open Short,09-04-2025 23:52
4,0x271b280974205ca63b716753467d5a371de622ab,85397101591,82617.625000,470685.70,5.69725,158.150382,0.000,SELL,Open Short,10-04-2025 00:09


In [105]:
agg_df_bitcoin['Timestamp IST'] = pd.to_datetime(agg_df_bitcoin['Timestamp IST'])
agg_df_bitcoin['trade_date'] = agg_df_bitcoin['Timestamp IST'].dt.date

/tmp/ipykernel_19823/2267319212.py:1: UserWarning:

Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.



In [106]:
agg_df_bitcoin.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5170 entries, 0 to 5169
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Account          5170 non-null   object        
 1   Order ID         5170 non-null   int64         
 2   Execution Price  5170 non-null   float64       
 3   Size USD         5170 non-null   float64       
 4   Size Tokens      5170 non-null   float64       
 5   Fee              5170 non-null   float64       
 6   Closed PnL       5170 non-null   float64       
 7   Side             5170 non-null   object        
 8   Direction        5170 non-null   object        
 9   Timestamp IST    5170 non-null   datetime64[ns]
 10  trade_date       5170 non-null   object        
dtypes: datetime64[ns](1), float64(5), int64(1), object(4)
memory usage: 444.4+ KB


In [107]:
agg_df_bitcoin.head()

,Account,Order ID,Execution Price,Size USD,Size Tokens,Fee,Closed PnL,Side,Direction,Timestamp IST,trade_date
0,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,88138871892,93700.000000,3456.59,0.03689,0.345657,0.000,SELL,Open Short,2025-04-23 05:12:00,2025-04-23
1,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,88141374064,93500.000000,3449.21,0.03689,0.344921,7.378,BUY,Close Short,2025-04-23 05:22:00,2025-04-23
2,0x271b280974205ca63b716753467d5a371de622ab,85391103553,82533.666667,460268.75,5.57673,154.650290,0.000,SELL,Open Short,2025-04-09 23:51:00,2025-04-09
3,0x271b280974205ca63b716753467d5a371de622ab,85391332312,82575.736842,780030.44,9.44631,262.090211,0.000,SELL,Open Short,2025-04-09 23:52:00,2025-04-09
4,0x271b280974205ca63b716753467d5a371de622ab,85397101591,82617.625000,470685.70,5.69725,158.150382,0.000,SELL,Open Short,2025-04-10 00:09:00,2025-04-10


In [108]:
market_sentiment['date'] = pd.to_datetime(market_sentiment['date']).dt.date

In [109]:
start_date = agg_df_bitcoin['trade_date'].min()
end_date = agg_df_bitcoin['trade_date'].max()

market_sentiment = market_sentiment[
    (market_sentiment['date'] >= start_date) &
    (market_sentiment['date'] <= end_date)
]

In [110]:
market_sentiment.shape

(513, 4)

**To ensure temporal alignment and avoid unnecessary data, the sentiment dataset was filtered to match the date range of the trading data before merging.**

In [111]:
final_bitcoin_data = agg_df_bitcoin.merge(market_sentiment, left_on='trade_date', right_on='date', how='left')

In [112]:
final_bitcoin_data['classification'].isnull().sum()

np.int64(0)

In [113]:
final_bitcoin_data.head(10)

,Account,Order ID,Execution Price,Size USD,Size Tokens,Fee,Closed PnL,Side,Direction,Timestamp IST,trade_date,timestamp,value,classification,date
0,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,88138871892,93700.000000,3456.59,0.03689,0.345657,0.000000,SELL,Open Short,2025-04-23 05:12:00,2025-04-23,1745386200,72,Greed,2025-04-23
1,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,88141374064,93500.000000,3449.21,0.03689,0.344921,7.378000,BUY,Close Short,2025-04-23 05:22:00,2025-04-23,1745386200,72,Greed,2025-04-23
2,0x271b280974205ca63b716753467d5a371de622ab,85391103553,82533.666667,460268.75,5.57673,154.650290,0.000000,SELL,Open Short,2025-04-09 23:51:00,2025-04-09,1744176600,18,Extreme Fear,2025-04-09
3,0x271b280974205ca63b716753467d5a371de622ab,85391332312,82575.736842,780030.44,9.44631,262.090211,0.000000,SELL,Open Short,2025-04-09 23:52:00,2025-04-09,1744176600,18,Extreme Fear,2025-04-09
4,0x271b280974205ca63b716753467d5a371de622ab,85397101591,82617.625000,470685.70,5.69725,158.150382,0.000000,SELL,Open Short,2025-04-10 00:09:00,2025-04-10,1744263000,39,Fear,2025-04-10
5,0x271b280974205ca63b716753467d5a371de622ab,85397181312,82598.666667,202916.30,2.45666,68.179864,0.000000,SELL,Open Short,2025-04-10 00:10:00,2025-04-10,1744263000,39,Fear,2025-04-10
6,0x271b280974205ca63b716753467d5a371de622ab,85397204065,82593.000000,66925.10,0.81030,22.486834,0.000000,SELL,Open Short,2025-04-10 00:10:00,2025-04-10,1744263000,39,Fear,2025-04-10
7,0x271b280974205ca63b716753467d5a371de622ab,85398764412,82269.000000,315745.14,3.83796,106.090361,1186.697232,BUY,Close Short,2025-04-10 00:15:00,2025-04-10,1744263000,39,Fear,2025-04-10
8,0x271b280974205ca63b716753467d5a371de622ab,85398834732,82181.000000,413971.98,5.03732,139.094583,2000.823504,BUY,Close Short,2025-04-10 00:15:00,2025-04-10,1744263000,39,Fear,2025-04-10
9,0x271b280974205ca63b716753467d5a371de622ab,85398901114,82113.333333,260588.78,3.17351,87.557826,1473.963202,BUY,Close Short,2025-04-10 00:15:00,2025-04-10,1744263000,39,Fear,2025-04-10


In [114]:
final_bitcoin_data.isnull().sum()

,0
Account,0
Order ID,0
Execution Price,0
Size USD,0
Size Tokens,0
Fee,0
Closed PnL,0
Side,0
Direction,0
Timestamp IST,0


In [115]:
final_bitcoin_data.duplicated().sum()

np.int64(0)

In [116]:
(final_bitcoin_data['Size USD'] <= 0).sum()

np.int64(0)

In [117]:
final_bitcoin_data['Closed PnL'].describe()

,Closed PnL
count,5170.000000
mean,171.430338
std,2497.922022
min,-59107.612824
25%,0.000000
50%,0.000000
75%,0.000000
max,43174.760153


In [118]:
final_bitcoin_data['classification'].value_counts()

,count
classification,
Greed,1675
Neutral,1330
Fear,1030
Extreme Greed,877
Extreme Fear,258


**As we want to understand trader performance so we will need to exclude all the open trades because pnl will always be zero if a trade is open.**

In [119]:
final_bitcoin_data = final_bitcoin_data[final_bitcoin_data['Direction'].isin(['Close Long', 'Close Short'])]

In [120]:
len(final_bitcoin_data)

1608

In [121]:
final_bitcoin_data.head(10)

,Account,Order ID,Execution Price,Size USD,Size Tokens,Fee,Closed PnL,Side,Direction,Timestamp IST,trade_date,timestamp,value,classification,date
1,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,88141374064,93500.000000,3449.21,0.03689,0.344921,7.378000,BUY,Close Short,2025-04-23 05:22:00,2025-04-23,1745386200,72,Greed,2025-04-23
7,0x271b280974205ca63b716753467d5a371de622ab,85398764412,82269.000000,315745.14,3.83796,106.090361,1186.697232,BUY,Close Short,2025-04-10 00:15:00,2025-04-10,1744263000,39,Fear,2025-04-10
8,0x271b280974205ca63b716753467d5a371de622ab,85398834732,82181.000000,413971.98,5.03732,139.094583,2000.823504,BUY,Close Short,2025-04-10 00:15:00,2025-04-10,1744263000,39,Fear,2025-04-10
9,0x271b280974205ca63b716753467d5a371de622ab,85398901114,82113.333333,260588.78,3.17351,87.557826,1473.963202,BUY,Close Short,2025-04-10 00:15:00,2025-04-10,1744263000,39,Fear,2025-04-10
10,0x271b280974205ca63b716753467d5a371de622ab,85399056815,82036.181818,274229.06,3.34276,92.140950,1810.058682,BUY,Close Short,2025-04-10 00:16:00,2025-04-10,1744263000,39,Fear,2025-04-10
11,0x271b280974205ca63b716753467d5a371de622ab,85399254845,82136.000000,218864.51,2.66466,73.538475,1178.312652,BUY,Close Short,2025-04-10 00:16:00,2025-04-10,1744263000,39,Fear,2025-04-10
12,0x271b280974205ca63b716753467d5a371de622ab,85402451670,82030.200000,111898.21,1.36413,37.597789,749.186976,BUY,Close Short,2025-04-10 00:27:00,2025-04-10,1744263000,39,Fear,2025-04-10
13,0x271b280974205ca63b716753467d5a371de622ab,85405429626,81976.333333,97337.68,1.18739,32.705460,714.848378,BUY,Close Short,2025-04-10 00:38:00,2025-04-10,1744263000,39,Fear,2025-04-10
14,0x271b280974205ca63b716753467d5a371de622ab,85405458748,81938.300000,276913.92,3.37952,93.043071,2160.757504,BUY,Close Short,2025-04-10 00:38:00,2025-04-10,1744263000,39,Fear,2025-04-10
20,0x271b280974205ca63b716753467d5a371de622ab,85452914712,82620.208333,463124.08,5.60557,155.609664,225.621766,BUY,Close Short,2025-04-10 05:28:00,2025-04-10,1744263000,39,Fear,2025-04-10


**Feature Engineering for EDA**

In [122]:
final_bitcoin_data['pnl_flag'] = final_bitcoin_data['Closed PnL'].apply(lambda x : 'Win' if x > 0 else ('Loss' if x < 0 else 'Breakeven'))

In [123]:
final_bitcoin_data['pnl_ratio'] = final_bitcoin_data['Closed PnL'] / final_bitcoin_data['Size USD']

In [124]:
final_bitcoin_data['position_type'] = final_bitcoin_data['Direction'].apply(
    lambda x: 'Long' if 'Long' in x else 'Short'
)

In [125]:
final_bitcoin_data = final_bitcoin_data.rename(
    columns={'classification': 'market_sentiment'}
)

In [126]:
final_bitcoin_data.duplicated().sum()

np.int64(0)

###***Exploratory Data Analysis***

**Objective:** To explore the relationship between trader performance and market sentiment, uncover hidden patterns, and deliver insights that can drive smarter trading strategies.

***Thought Process before EDA***

Before jumping into the analysis, I tried to break down the problem and think about what factors could actually influence trader performance.

The main idea was to explore how market sentiment affects profitability, and whether traders behave differently under Fear vs Greed conditions.

Some of the key questions I wanted to explore were:

Does market sentiment directly impact PnL? For example, do traders perform better during Fear compared to Greed?
How does trade size (Size USD) play a role — does putting more capital lead to better returns, or just higher risk?
What is the distribution of winning vs losing trades across different sentiment phases?
Are Long or Short positions generally more profitable, and does that change with sentiment?
Does combining factors give deeper insights — for example,
Are Long trades more profitable during Fear?
Do Short trades fail during Greed?
Is there any relationship between sentiment, position type, and capital allocation together that explains performance better?

The goal was not just to look at these factors individually, but also to see how they interact with each other and whether they reveal any consistent trading patterns.

In [127]:
final_bitcoin_data.head()

,Account,Order ID,Execution Price,Size USD,Size Tokens,Fee,Closed PnL,Side,Direction,Timestamp IST,trade_date,timestamp,value,market_sentiment,date,pnl_flag,pnl_ratio,position_type
1,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,88141374064,93500.000000,3449.21,0.03689,0.344921,7.378000,BUY,Close Short,2025-04-23 05:22:00,2025-04-23,1745386200,72,Greed,2025-04-23,Win,0.002139,Short
7,0x271b280974205ca63b716753467d5a371de622ab,85398764412,82269.000000,315745.14,3.83796,106.090361,1186.697232,BUY,Close Short,2025-04-10 00:15:00,2025-04-10,1744263000,39,Fear,2025-04-10,Win,0.003758,Short
8,0x271b280974205ca63b716753467d5a371de622ab,85398834732,82181.000000,413971.98,5.03732,139.094583,2000.823504,BUY,Close Short,2025-04-10 00:15:00,2025-04-10,1744263000,39,Fear,2025-04-10,Win,0.004833,Short
9,0x271b280974205ca63b716753467d5a371de622ab,85398901114,82113.333333,260588.78,3.17351,87.557826,1473.963202,BUY,Close Short,2025-04-10 00:15:00,2025-04-10,1744263000,39,Fear,2025-04-10,Win,0.005656,Short
10,0x271b280974205ca63b716753467d5a371de622ab,85399056815,82036.181818,274229.06,3.34276,92.140950,1810.058682,BUY,Close Short,2025-04-10 00:16:00,2025-04-10,1744263000,39,Fear,2025-04-10,Win,0.006601,Short


####***Chart 1 - Trader Performance Vs Market Sentiment***

In [128]:
sentiment_pnl = final_bitcoin_data.groupby('market_sentiment')['Closed PnL'].mean().reset_index()

fig = px.bar(
    sentiment_pnl,
    x='market_sentiment',
    y='Closed PnL',
    color = 'market_sentiment',
    title = 'Trader Performance Vs Market Sentiment',
    text_auto = True
)
fig.show()

**Insights**

1.   Profitability peaks during Fear, indicating better trade outcomes in cautious market conditions.
2.   Greed and Extreme Greed phases show relatively lower returns, suggesting overconfidence impacts performance negatively.
3. Neutral markets provide stable but less aggressive profit opportunities.





####***Chart 2 - Market Sentiment Vs PnL Ratio(Efficiency)***

In [129]:
senitment_pnl_ratio = final_bitcoin_data.groupby('market_sentiment')['pnl_ratio'].mean().reset_index()
fig = px.bar(
    senitment_pnl_ratio,
    x='market_sentiment',
    y='pnl_ratio',
    color='market_sentiment',
    title = 'Market Sentiment Vs PnL Ratio(Efficiency)',
    text_auto = True
)
fig.show()

**Insights**

1. Trading efficiency is highest during Fear, showing better capital utilization in bearish sentiment.

2. Greed is the only phase with negative returns, highlighting poor trade quality during optimistic conditions.

3. Extreme Greed shows selective high returns but lacks consistency.

In [130]:
final_bitcoin_data.groupby('market_sentiment')['Closed PnL'].std()

,Closed PnL
market_sentiment,
Extreme Fear,6064.747481
Extreme Greed,1028.391627
Fear,6389.858561
Greed,3333.880181
Neutral,5841.004574


**Insights**

Trading performance is strongly influenced by market sentiment. Fear-driven markets offer the highest profitability and risk-adjusted returns, albeit with higher volatility, indicating strong opportunity zones for informed traders. In contrast, Greed phases show negative risk-adjusted returns, suggesting that traders tend to overtrade or make suboptimal decisions during optimistic market conditions

####***Chart 3 - Market Sentiment vs Win/Loss Distribution***

In [131]:
df = final_bitcoin_data[final_bitcoin_data['pnl_flag'] != 'Breakeven']
win_rate = df.groupby('market_sentiment')['pnl_flag'].value_counts(normalize=True).reset_index()
fig =  px.bar(
    win_rate,
    x='market_sentiment',
    y='proportion',
    color = 'pnl_flag',
    barmode='group',
    title='Market Sentiment vs Win/Loss Distribution',
)

fig.show()

**Insights**

1. Win rates are highest during Fear and Extreme Greed, indicating that traders
achieve higher success rates in both strongly negative and strongly positive market phases.

2. Greed shows a significantly lower win rate (~55%), suggesting that moderately optimistic conditions lead to poorer trade decisions and higher losses.

3. Extreme Fear has comparatively higher loss proportion, indicating increased volatility and risk despite decent win rates.

####***Chart 4 - Market Sentiment vs Position Type Distribution***

In [132]:
position_dist = final_bitcoin_data.groupby('market_sentiment')['position_type'].value_counts(normalize=True).reset_index()

fig = px.bar(
    position_dist,
    x='market_sentiment',
    y='proportion',
    color='position_type',
    barmode='group',
    title='Market Sentiment vs Position Type Distribution'
)

fig.show()

**Insights**

1. Traders predominantly take Long positions during Fear and Neutral phases, indicating a tendency to buy during perceived recovery or stable conditions.

2. In Greed, Short positions slightly dominate, suggesting traders anticipate potential market corrections after bullish runs.

3. Extreme Greed shows a near-balanced distribution of Long and Short positions, reflecting uncertainty and mixed expectations at market peaks.

####***Chart 5 - Position Type vs Average PnL***

In [133]:
pos_pnl = final_bitcoin_data.groupby('position_type')['Closed PnL'].mean().reset_index()

fig = px.bar(
    pos_pnl,
    x='position_type',
    y='Closed PnL',
    color = 'position_type',
    title='Position Type vs Average PnL',
    text_auto=True
)

fig.show()

**Insights**
1. Long positions generate significantly higher profits, while Short positions result in overall losses, indicating a strong directional bias in trader performance.

2. The negative PnL for Shorts suggests that traders struggle to capitalize on downward market movements.

3. This reflects a bullish market structure during the observed period, where long trades consistently outperform.

####***Chart 6 - Position Type vs PnL Ratio***

In [134]:
pos_ratio = final_bitcoin_data.groupby('position_type')['pnl_ratio'].mean().reset_index()

fig = px.bar(
    pos_ratio,
    x='position_type',
    y='pnl_ratio',
    title='Position Type vs PnL Ratio',
    text_auto=True
)

fig.show()

**Insights**

1. Long positions show positive trading efficiency, delivering better returns relative to capital deployed.
2. Short positions have a negative PnL ratio, indicating inefficient trades and consistent value erosion.
3. This suggests that traders are more effective in upward-trending markets than in bearish conditions.

####***Chart 7 - Market Sentiment vs Average Trade Size***

In [135]:
size_sentiment = final_bitcoin_data.groupby('market_sentiment')['Size USD'].mean().reset_index()

fig = px.bar(
    size_sentiment,
    x='market_sentiment',
    y='Size USD',
    color = 'market_sentiment',
    title='Market Sentiment vs Average Trade Size',
    text_auto=True
)

fig.show()

**Insights**

1. The highest capital deployment is observed during Fear, indicating that traders allocate more funds when markets are down, possibly to capture discounted opportunities.

2. Extreme Greed shows the lowest trade size, suggesting cautious participation or reduced conviction at market peaks.

3. Moderate allocation during Neutral and Greed phases reflects balanced but less aggressive trading behavior compared to fear-driven markets.

####***Analysis: Market Sentiment Vs Positon Type Vs PNL Flag***

In [136]:
df.groupby(['market_sentiment', 'position_type'])['pnl_flag'].value_counts(normalize=True)

market_sentiment  position_type  pnl_flag
Extreme Fear      Long           Win         0.712329
                                 Loss        0.287671
                  Short          Win         0.888889
                                 Loss        0.111111
Extreme Greed     Long           Win         0.884146
                                 Loss        0.115854
                  Short          Win         0.861111
                                 Loss        0.138889
Fear              Long           Win         0.876190
                                 Loss        0.123810
                  Short          Win         0.881579
                                 Loss        0.118421
Greed             Long           Win         0.889610
                                 Loss        0.110390
                  Short          Loss        0.742297
                                 Win         0.257703
Neutral           Long           Win         0.919786
                                 Loss        0.080214
                  Short          Win         0.577465
                                 Loss        0.422535
Name: proportion, dtype: float64

**Insights**

1. Long positions maintain consistently high win rates across all sentiments, especially in Neutral and Greed, indicating strong reliability of long-biased strategies.

2. Short positions are highly sentiment-dependent, performing well during Fear and Extreme Fear, but collapsing in Greed (very low win rate), highlighting poor timing in bullish conditions.

3. This suggests that short trades require precise sentiment timing, while long trades remain more robust across market phases.

####***Chart 8 - Sentiment vs Position Type vs Avg PnL***

In [137]:
import plotly.graph_objects as go

heatmap_data = final_bitcoin_data.groupby(
    ['market_sentiment', 'position_type']
)['Closed PnL'].mean().unstack()

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=heatmap_data.columns,
    y=heatmap_data.index,
    colorscale='RdYlGn',
    text=heatmap_data.values,
    texttemplate="%{text:.2f}",  # 👈 shows numbers
    colorbar_title="Avg PnL"
))

fig.update_layout(
    title="Sentiment vs Position Type vs Avg PnL",
    xaxis_title="Position Type",
    yaxis_title="Market Sentiment"
)

fig.show()

**Insights**

1. Long positions dominate profitability across most sentiments, especially during Fear, reinforcing a strong long-bias in trader success.

2. Short positions are highly inconsistent, generating profits in Fear, Neutral, and Extreme Greed, but leading to significant losses in Extreme Fear and Greed.

3. This indicates that short trades require precise sentiment alignment, while long trades remain more stable and reliable across market conditions.

###***🧠 Final Insights & Strategy Recommendations***

**🔍 1. Relationship Between Market Sentiment & Trader Performance**

Trader performance is strongly influenced by market sentiment, with the highest profitability and efficiency observed during Fear conditions.
Contrary to common intuition, Greed and Extreme Greed phases do not consistently yield higher profits, and in some cases (Greed), result in poor performance and negative returns.
This indicates that market sentiment acts as a behavioral signal, impacting decision quality rather than simply reflecting market direction.

**📊 2. Behavioral Patterns in Trading**

Traders tend to exhibit reactive behavior, increasing Long exposure during Fear and Neutral phases, and adjusting positions based on prevailing sentiment rather than anticipating reversals.
Herd behavior is evident during Greed, where traders enter positions late, leading to reduced profitability and lower win rates.
Capital allocation is highest during Fear, suggesting a “buy-the-dip” mindset, where traders deploy more funds in perceived undervalued conditions.

**⚖️ 3. Trade Direction Insights (Long vs Short)**

Long positions consistently outperform Short positions across most sentiment phases, both in terms of total profitability and trading efficiency.
Short trades are highly sentiment-dependent, performing well in Fear-driven markets but failing significantly during Greed phases.
This highlights that long strategies are more robust, while short strategies require precise timing and sentiment alignment.

**📉 4. Risk & Efficiency Observations**
Despite high win rates in certain sentiments (e.g., Extreme Greed), overall profitability does not always align, indicating that win rate alone is not a reliable performance metric.
Negative PnL ratios during Greed suggest inefficient capital usage and poor trade quality, likely driven by overconfidence.
The presence of losses in extreme conditions (e.g., Extreme Fear for Shorts) highlights increased volatility and risk exposure.

**🚀 5. Key Trading Strategy Implications**

Based on the observed patterns, the following strategy insights can be derived:

Prioritize trading during Fear phases, where both profitability and efficiency are highest.
Avoid or limit trading during Greed phases, as they are associated with poor performance and higher losses.
Favor Long positions as a primary strategy, given their consistent profitability across sentiments.
Use Short positions selectively, particularly in Fear or correction phases, with strong risk management.
Focus on capital preservation and loss minimization, rather than relying solely on win rates.

###***Final Conclusion***

This analysis demonstrates that trader performance is not random but heavily influenced by market sentiment and behavioral biases.

Successful trading strategies should therefore:

Incorporate sentiment-aware decision making
Avoid emotionally driven market phases
Align trade direction with broader sentiment trends

Ultimately, combining data-driven insights with disciplined strategy execution can significantly enhance trading performance and reduce risk exposure.